<a href="https://colab.research.google.com/github/Amruth-U-tech/DL-Journey/blob/main/08-LLMs/LLM-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers torch

from transformers import AutoTokenizer, AutoModelForCausalLM  #importing predefined functions for tokenizer and model package with diff maodels in it
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"  #this is a library in transformers

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name,device_map="auto",
                                             torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()

print("using model:",model_name)
print("using device:",device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

using model: mistralai/Mistral-7B-Instruct-v0.2
using device: cpu


In [2]:
def answer_question(prompt,max_new_tokens=64):
  inputs = tokenizer(prompt, return_tensors="pt").to(device)
  with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
  return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

In [3]:
questions = [{"domain":"math",
              "question":"what is 27+4",
              "answer":"31"},
             {"domain":"science",
              "question":"what is the boiling point of water",
              "answer":"100"},
             {"domain":"science",
              "question":"which organ pumps the blood",
              "answer":"heart"},
             {"domain":"geography",
              "question":"what is the capital of india",
              "answer":"Delhi"},
             {"domain":"history",
              "question":"who was the first person to go on moon",
              "answer":"Neil Armstrong"},
             {"domain":"math",
              "question":"what is sin(90) where 90 is in degrees",
              "answer":"1"},]

In [4]:
import textwrap

def normalise(s:str):
  return s.lower().strip()
num_correct = 0

for i,item in enumerate(questions, start=1):
  domain = item["domain"]
  q = item["question"]
  answer = item["answer"]
  prompt = f"Answer the following question in breif and clearly.\n\nQuestion:{q}\n\nAnswer:"
  pred_ = answer_question(prompt)

  answer_norm = normalise(answer)
  pred__norm = normalise(pred_)
  is_correct = answer_norm in pred__norm
  if is_correct:
    num_correct += 1

  print("="*100)
  print(f"Q{i}|doamin:{domain}")
  print(f"Question:")
  print(textwrap.fill(q,width=90))
  print(f"model answer:{pred_}")
  print(f"Correct answer:{answer}")
  print("marked as:","✅ correct" if is_correct else "❌ incorrect")


print("\n"+"="*100)
print(f"you got {num_correct} correct out of {len(questions)}")




Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q1|doamin:math
Question:
what is 27+4
model answer:Answer the following question in breif and clearly.

Question:what is 27+4

Answer: The sum of 27 and 4 is 31.
Correct answer:31
marked as: ✅ correct


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q2|doamin:science
Question:
what is the boiling point of water
model answer:Answer the following question in breif and clearly.

Question:what is the boiling point of water

Answer: The boiling point of water is 100 degrees Celsius or 212 degrees Fahrenheit at standard atmospheric pressure. This temperature marks the point where water transitions from a liquid state to a gas state, forming steam.
Correct answer:100
marked as: ✅ correct


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q3|doamin:science
Question:
which organ pumps the blood
model answer:Answer the following question in breif and clearly.

Question:which organ pumps the blood

Answer: The heart is the organ that pumps blood throughout the body. It is a muscular organ located in the chest cavity, and it contracts and relaxes to push blood through the circulatory system. The heart has four chambers: two atria and two ventricles. Deoxygenated blood from the
Correct answer:heart
marked as: ✅ correct


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q4|doamin:geography
Question:
what is the capital of india
model answer:Answer the following question in breif and clearly.

Question:what is the capital of india

Answer: The capital city of India is New Delhi. It is a city that holds great historical and cultural significance and serves as the political and administrative hub of the country. New Delhi was established as the capital in 1931, replacing Calcutta and Bombay as the capital of British India. Today, it is
Correct answer:Delhi
marked as: ✅ correct


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q5|doamin:history
Question:
who was the first person to go on moon
model answer:Answer the following question in breif and clearly.

Question:who was the first person to go on moon

Answer: The first person to walk on the moon was Neil Armstrong on July 20, 1969. He made this historic step during the Apollo 11 mission, which was a spaceflight by NASA, the United States' national space agency. Armstrong famously declared, "That
Correct answer:Neil Armstrong
marked as: ✅ correct
Q6|doamin:math
Question:
what is sin(90) where 90 is in degrees
model answer:Answer the following question in breif and clearly.

Question:what is sin(90) where 90 is in degrees

Answer: The value of sin(90) degrees is 0. This is because the sine function gives the ratio of the length of the side opposite an angle in a right triangle to the length of the hypotenuse. In a right triangle, the angle of 90 degrees is a right angle, and
Correct answer:1
marked as: ❌ incorrect

you got 5 correct out of 6
